# SEC Form 13F filing-selection audit

This notebook audits the filing decisions produced by `scripts/03_select_sec_13f_filings.py` across the complete configured sample.

It checks:

1. whether the saved decision and selected-filing tables agree;
2. whether every selected filing satisfies the temporal and report-type rules;
3. whether each selected manager-quarter has exactly one base filing; and
4. how amendments, notices, disagreements, and unresolved filings were treated.

The notebook does not repeat or modify the selection algorithm.

## 1. Setup and inputs

In [1]:
import sys
from pathlib import Path

import pandas as pd


project_root = Path.cwd().resolve()

while not (project_root / "config" / "paths.yaml").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not locate the project root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.configuration import get_paths_config, get_sec_13f_config


paths = get_paths_config()
source = get_sec_13f_config()
selection_directory = paths.interim / "sec_13f" / "filing_selection"
filing_decisions_path = selection_directory / "filing_decisions.parquet"
selected_filings_path = selection_directory / "selected_filings.parquet"

for path in (filing_decisions_path, selected_filings_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required filing-selection output not found: {path}")

filing_decisions = pd.read_parquet(filing_decisions_path)
selected_filings = pd.read_parquet(selected_filings_path)
group_key = ["CIK", "report_period"]

print(f"Filing decisions: {len(filing_decisions):,}")
print(f"Selected filings: {len(selected_filings):,}")
print(
    "Configured report periods: "
    f"{source.report_period_start_date} to {source.report_period_end_date}"
)

Filing decisions: 397,833
Selected filings: 279,051
Configured report periods: 2013-06-30 to 2026-03-31


## 2. Sample coverage and decisions

Every assessed filing receives one explicit decision. Selected filings consist only of retained base filings and valid `NEW HOLDINGS` additions.

In [2]:
decision_summary = (
    filing_decisions["selection_decision"]
    .value_counts(dropna=False)
    .rename_axis("selection_decision")
    .reset_index(name="filings")
)
decision_summary["percent"] = (
    decision_summary["filings"] / len(filing_decisions) * 100
).round(2)

selected_periods = pd.to_datetime(selected_filings["report_period"])

coverage_summary = pd.DataFrame(
    {
        "measure": [
            "Filings assessed",
            "Distinct report periods assessed",
            "Selected filings",
            "Selected base filings",
            "Selected additions",
            "Selected manager-quarters",
        ],
        "count": [
            len(filing_decisions),
            filing_decisions["report_period"].nunique(),
            len(selected_filings),
            selected_filings["selection_decision"].eq("SELECTED_BASE").sum(),
            selected_filings["selection_decision"].eq("SELECTED_ADDITION").sum(),
            selected_filings.groupby(group_key).ngroups,
        ],
    }
)

display(coverage_summary.style.format({"count": "{:,}"}))
display(decision_summary.style.format({"filings": "{:,}"}))

print()
print(
    "Selected report-period range: "
    f"{selected_periods.min().date()} to {selected_periods.max().date()}"
)

,measure,count
0,Filings assessed,"397,833"
1,Distinct report periods assessed,98
2,Selected filings,"279,051"
3,Selected base filings,"278,656"
4,Selected additions,395
5,Selected manager-quarters,"278,656"


,selection_decision,filings,percent
0,SELECTED_BASE,"278,656",70.040000
1,NOTICE_EXCLUDED,"80,447",20.220000
2,FILED_AFTER_DEADLINE,"33,731",8.480000
3,SUPERSEDED,"3,266",0.820000
4,OUTSIDE_SAMPLE_PERIOD,"1,312",0.330000
5,SELECTED_ADDITION,395,0.100000
6,UNRESOLVED,15,0.000000
7,CLASSIFICATION_DISAGREEMENT,11,0.000000



Selected report-period range: 2013-06-30 to 2026-03-31


## 3. Selection-integrity checks

These checks test the saved outputs against the rules implemented by Script 03. A valid selection should report zero violations in every row.

In [3]:
expected_selected_accessions = set(
    filing_decisions.loc[
        filing_decisions["is_selected"],
        "ACCESSION_NUMBER",
    ]
)
saved_selected_accessions = set(selected_filings["ACCESSION_NUMBER"])

base_counts = (
    selected_filings.assign(
        is_selected_base=selected_filings["selection_decision"].eq(
            "SELECTED_BASE"
        )
    )
    .groupby(group_key)["is_selected_base"]
    .sum()
)

selected_bases = selected_filings.loc[
    selected_filings["selection_decision"].eq("SELECTED_BASE"),
    group_key + ["amendment_number"],
].rename(columns={"amendment_number": "base_amendment_number"})

selected_additions = selected_filings.loc[
    selected_filings["selection_decision"].eq("SELECTED_ADDITION")
].merge(
    selected_bases,
    on=group_key,
    how="left",
    validate="many_to_one",
)

valid_selected_decisions = {"SELECTED_BASE", "SELECTED_ADDITION"}

integrity_checks = pd.DataFrame(
    {
        "check": [
            "Duplicate accessions in filing decisions",
            "Duplicate accessions in selected filings",
            "Mismatch between selection flag and saved selected table",
            "Selected filings outside the configured sample",
            "Selected filings with non-quarter-end report periods",
            "Selected filings submitted after the cutoff",
            "Selected filings not classified as holdings candidates",
            "Selected filings with an invalid decision",
            "Selected manager-quarters without exactly one base",
            "Selected additions not labeled NEW HOLDINGS",
            "Selected additions without a higher amendment number",
        ],
        "violations": [
            filing_decisions["ACCESSION_NUMBER"].duplicated().sum(),
            selected_filings["ACCESSION_NUMBER"].duplicated().sum(),
            len(expected_selected_accessions ^ saved_selected_accessions),
            (~selected_filings["is_in_sample_period"]).sum(),
            (~selected_filings["is_standard_quarter_end"]).sum(),
            (~selected_filings["is_filed_by_deadline"]).sum(),
            (~selected_filings["is_holdings_candidate"]).sum(),
            (~selected_filings["selection_decision"].isin(
                valid_selected_decisions
            )).sum(),
            base_counts.ne(1).sum(),
            (~selected_additions["amendment_kind"].eq("NEW HOLDINGS")).sum(),
            (~selected_additions["amendment_number"].gt(
                selected_additions["base_amendment_number"]
            )).sum(),
        ],
    }
)

display(integrity_checks.style.format({"violations": "{:,}"}))
print()
print(f"Total integrity violations: {integrity_checks['violations'].sum():,}")

,check,violations
0,Duplicate accessions in filing decisions,0
1,Duplicate accessions in selected filings,0
2,Mismatch between selection flag and saved selected table,0
3,Selected filings outside the configured sample,0
4,Selected filings with non-quarter-end report periods,0
5,Selected filings submitted after the cutoff,0
6,Selected filings not classified as holdings candidates,0
7,Selected filings with an invalid decision,0
8,Selected manager-quarters without exactly one base,0
9,Selected additions not labeled NEW HOLDINGS,0



Total integrity violations: 0


## 4. Point-in-time timing

The filing lag is the number of calendar days between the reported quarter-end and the public filing date. The cutoff is 45 calendar days after quarter-end, moved forward when it falls on a weekend or U.S. federal holiday.

In [4]:
lag_summary = (
    selected_filings.groupby("selection_decision")["filing_lag_days"]
    .agg(
        filings="count",
        minimum="min",
        median="median",
        maximum="max",
    )
    .reset_index()
)

deadline_extensions = (
    pd.to_datetime(selected_filings["filing_deadline"])
    - pd.to_datetime(selected_filings["report_period"])
).dt.days

display(lag_summary.style.format({"filings": "{:,}", "median": "{:.1f}"}))

print()
print(
    "Quarterly cutoffs later than 45 calendar days: "
    f"{selected_filings.loc[deadline_extensions.gt(45), 'report_period'].nunique():,}"
)
print(f"Latest selected filing lag: {selected_filings['filing_lag_days'].max()} days")

,selection_decision,filings,minimum,median,maximum
0,SELECTED_ADDITION,395,2,42.0,48
1,SELECTED_BASE,"278,656",0,40.0,48



Quarterly cutoffs later than 45 calendar days: 13
Latest selected filing lag: 48 days


## 5. Amendment resolution

Multiple eligible holdings filings for the same manager-quarter are the cases in which the ordering and amendment rules affect the selected portfolio.

In [5]:
temporally_eligible = (
    filing_decisions["is_in_sample_period"]
    & filing_decisions["is_filed_by_deadline"]
)
eligible_holdings = filing_decisions.loc[
    temporally_eligible & filing_decisions["is_holdings_candidate"]
]

eligible_group_sizes = eligible_holdings.groupby(group_key).size()
multi_filing_groups = int(eligible_group_sizes.gt(1).sum())

resolution_summary = pd.DataFrame(
    {
        "measure": [
            "Eligible holdings filings",
            "Eligible manager-quarters",
            "Manager-quarters with multiple eligible filings",
            "Selected restatement bases",
            "Selected new-holdings additions",
            "Unresolved holdings filings",
        ],
        "count": [
            len(eligible_holdings),
            eligible_holdings.groupby(group_key).ngroups,
            multi_filing_groups,
            (
                selected_filings["selection_decision"].eq("SELECTED_BASE")
                & selected_filings["amendment_kind"].eq("RESTATEMENT")
            ).sum(),
            selected_filings["selection_decision"].eq(
                "SELECTED_ADDITION"
            ).sum(),
            filing_decisions["selection_decision"].eq("UNRESOLVED").sum(),
        ],
    }
)

display(resolution_summary.style.format({"count": "{:,}"}))

unresolved_filings = filing_decisions.loc[
    filing_decisions["selection_decision"].eq("UNRESOLVED"),
    [
        "ACCESSION_NUMBER",
        "CIK",
        "report_period",
        "filing_date",
        "amendment_kind",
    ],
]

if not unresolved_filings.empty:
    print()
    print("Unresolved filings")
    display(unresolved_filings.reset_index(drop=True))

,measure,count
0,Eligible holdings filings,"282,332"
1,Eligible manager-quarters,"278,671"
2,Manager-quarters with multiple eligible filings,"3,527"
3,Selected restatement bases,"3,159"
4,Selected new-holdings additions,395
5,Unresolved holdings filings,15



Unresolved filings


,ACCESSION_NUMBER,CIK,report_period,filing_date,amendment_kind
0,0001104659-14-059287,0001085867,2014-06-30,2014-08-11,NEW HOLDINGS
1,0001104659-15-058626,0001085867,2015-06-30,2015-08-11,NEW HOLDINGS
2,0001104659-16-138180,0001085867,2016-06-30,2016-08-09,NEW HOLDINGS
3,0001104659-14-077513,0001085867,2014-09-30,2014-11-06,NEW HOLDINGS
4,0001104659-15-075162,0001085867,2015-09-30,2015-11-03,NEW HOLDINGS
5,0001104659-16-156124,0001085867,2016-09-30,2016-11-10,NEW HOLDINGS
6,0001104659-15-005982,0001085867,2014-12-31,2015-02-02,NEW HOLDINGS
7,0001104659-16-094914,0001085867,2015-12-31,2016-02-09,NEW HOLDINGS
8,0001104659-14-037523,0001085867,2014-03-31,2014-05-12,NEW HOLDINGS
9,0001104659-15-036629,0001085867,2015-03-31,2015-05-11,NEW HOLDINGS


## 6. Source-signal disagreements

The audit keeps disagreement flags visible because the SEC submission type, cover-page report type, and amendment fields do not always convey identical classifications.

In [6]:
in_sample_filings = filing_decisions.loc[
    filing_decisions["is_in_sample_period"]
]

disagreement_summary = pd.DataFrame(
    {
        "issue": [
            "Amendment-signal disagreements",
            "Classification disagreements",
            "Notices excluded",
        ],
        "in_sample_filings": [
            in_sample_filings["amendment_signal_disagreement"].sum(),
            in_sample_filings["classification_disagreement"].sum(),
            in_sample_filings["selection_decision"].eq(
                "NOTICE_EXCLUDED"
            ).sum(),
        ],
        "selected_filings": [
            selected_filings["amendment_signal_disagreement"].sum(),
            selected_filings["classification_disagreement"].sum(),
            0,
        ],
    }
)

display(
    disagreement_summary.style.format(
        {
            "in_sample_filings": "{:,}",
            "selected_filings": "{:,}",
        }
    )
)

selected_amendment_disagreements = (
    selected_filings.loc[selected_filings["amendment_signal_disagreement"]]
    .groupby(
        [
            "SUBMISSIONTYPE",
            "ISAMENDMENT",
            "AMENDMENTTYPE",
            "selection_decision",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="filings")
)

if not selected_amendment_disagreements.empty:
    print()
    print("Selected amendment-signal disagreements")
    display(selected_amendment_disagreements)

,issue,in_sample_filings,selected_filings
0,Amendment-signal disagreements,95,21
1,Classification disagreements,11,0
2,Notices excluded,"80,447",0



Selected amendment-signal disagreements


,SUBMISSIONTYPE,ISAMENDMENT,AMENDMENTTYPE,selection_decision,filings
0,13F-HR,Y,RESTATEMENT,SELECTED_BASE,21


## 7. Quarterly coverage

The final table shows how many base portfolios and supplemental additions enter each report quarter.

In [7]:
quarterly_coverage = (
    selected_filings.assign(
        selected_bases=selected_filings["selection_decision"].eq(
            "SELECTED_BASE"
        ),
        selected_additions=selected_filings["selection_decision"].eq(
            "SELECTED_ADDITION"
        ),
    )
    .groupby("report_period", as_index=False)
    .agg(
        selected_bases=("selected_bases", "sum"),
        selected_additions=("selected_additions", "sum"),
    )
    .sort_values("report_period")
)

display(
    quarterly_coverage.style.format(
        {
            "selected_bases": "{:,}",
            "selected_additions": "{:,}",
        }
    )
)

print()
print(f"Report quarters selected: {len(quarterly_coverage):,}")
print(
    "Base portfolios per quarter: "
    f"{quarterly_coverage['selected_bases'].min():,} to "
    f"{quarterly_coverage['selected_bases'].max():,}"
)

,report_period,selected_bases,selected_additions
0,2013-06-30 00:00:00,"3,051",5
1,2013-09-30 00:00:00,"3,227",9
2,2013-12-31 00:00:00,"3,552",6
3,2014-03-31 00:00:00,"3,593",4
4,2014-06-30 00:00:00,"3,570",11
5,2014-09-30 00:00:00,"3,621",10
6,2014-12-31 00:00:00,"3,862",12
7,2015-03-31 00:00:00,"3,929",3
8,2015-06-30 00:00:00,"3,934",9
9,2015-09-30 00:00:00,"3,966",8



Report quarters selected: 52
Base portfolios per quarter: 3,051 to 8,571


## 8. Interpretation

- **The saved selection is internally consistent.** All integrity checks return zero violations: accession numbers are unique, the two saved tables agree, all selected filings satisfy the report-period, cutoff, and holdings-classification rules, and every selected manager-quarter has exactly one base filing.
- **The point-in-time restriction is consequential.** Of 397,833 assessed filings, 33,731 were filed after their applicable cutoff and were excluded. Selected filings have lags of up to 48 days because 13 quarter-specific 45-day cutoffs were moved to the next business day.
- **Amendments are resolved explicitly.** The final sample contains 278,656 base filings and 395 supplemental `NEW HOLDINGS` amendments. The amendment rules matter for 3,527 manager-quarters with multiple eligible holdings filings. Fifteen `NEW HOLDINGS` filings have no eligible base and are therefore correctly classified as unresolved rather than used as complete portfolios.
- **Source-signal disagreements are limited and handled conservatively.** Eleven holdings-classification disagreements are excluded. The 21 selected amendment-signal disagreements are all filings whose submission type is `13F-HR` while the cover page identifies them as `Y` and `RESTATEMENT`; treating any positive amendment signal as an amendment preserves these restatements.
- **Quarterly coverage is complete but expands over time.** The selection covers all 52 quarters from 2013 Q2 through 2026 Q1. The number of base portfolios rises from 3,051 to 8,571 per quarter, which reflects changing filing coverage and should be considered when interpreting longitudinal network statistics.

The selected accessions are therefore suitable for constructing and auditing the manager-security position universe in the next stage.